# 🛒 Smart Shopping Assistant
1. An AI agent where users paste a product description.
2. Estimates a fair price using a fine-tuned pricing model.
3. Finds real listings via a web search tool.
Compares listing prices with the estimated price.
4. Returns a verdict: **Overpriced**, **Fair**, or **A Bargain**.

In [1]:
import os
import re
import json
from typing import List, Optional

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from langchain_anthropic import ChatAnthropic
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, BaseMessage

In [2]:
load_dotenv(override=True)

FINE_TUNED_MODEL = "ft:gpt-4.1-nano-2025-04-14:personal:pricer:DFJJ866N"
AGENT_MODEL = "claude-haiku-4-5"

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
llm = ChatAnthropic(model=AGENT_MODEL, temperature=0.0, max_tokens=1800)
web_search_tool = DuckDuckGoSearchRun()

In [3]:
@tool
def estimate_price(product_description: str) -> str:
    """Estimate the fair market price of a product using the fine-tuned pricing model.
    Pass the full product description or title and details."""
    response = openai_client.chat.completions.create(
        model=FINE_TUNED_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    "Estimate the price of this product. Respond with the price, no explanation\n\n"
                    + product_description
                ),
            }
        ],
        max_tokens=20,
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()


@tool
def web_search(query: str) -> str:
    """Search the web for real product listings, current prices, and availability."""
    return web_search_tool.run(query)


@tool
def compare_prices(estimated_price: float, listing_price: float) -> str:
    """Compare the model's estimated fair price against a real listing price and return a deal verdict.
    Both prices should be plain numbers in USD (e.g. 49.99)."""
    if listing_price <= 0 or estimated_price <= 0:
        return "Unable to compare: one or both prices are invalid."

    ratio = listing_price / estimated_price
    diff = listing_price - estimated_price
    direction = "above" if diff > 0 else "below"
    pct = abs((ratio - 1) * 100)

    if ratio <= 0.80:
        verdict = "Great deal! The listing price is significantly below fair market value."
    elif ratio <= 0.95:
        verdict = "Good deal. The listing price is slightly below fair market value."
    elif ratio <= 1.05:
        verdict = "Fair price. The listing matches the estimated market value."
    elif ratio <= 1.20:
        verdict = "Slightly overpriced. You may find it cheaper elsewhere."
    else:
        verdict = "Overpriced. Consider looking for a better deal."

    return (
        f"Estimated fair price: ${estimated_price:.2f}\n"
        f"Listing price: ${listing_price:.2f}\n"
        f"Difference: ${abs(diff):.2f} ({pct:.1f}% {direction} estimate)\n"
        f"Verdict: {verdict}"
    )


tools = [estimate_price, web_search, compare_prices]
agent_llm = llm.bind_tools(tools)

In [4]:
SYSTEM_PROMPT = """
You are a Smart Shopping Assistant. Your job is to help users decide if a product is worth buying at a given price.

When a user describes a product or asks about a deal, follow these steps in order:
1. Call `estimate_price` with the product description to get the fair market value.
2. Call `web_search` to find real current listings and prices for that product.
3. Extract a specific listing price from the web search results.
4. Call `compare_prices` with the estimated price and the listing price (both as plain numbers).
5. Summarise your findings clearly: the estimated value, what real listings show, and a deal recommendation.

If the user just asks a general shopping question (not about a specific product), answer directly without calling tools.
Be concise, helpful, and friendly.
"""


def handle_tool_call(tool_call: dict) -> ToolMessage:
    """Execute one tool call and return a LangChain ToolMessage."""
    tool_name = tool_call.get("name", "")
    tool_args = tool_call.get("args", {})
    tool_call_id = tool_call.get("id", "")

    if tool_name == "estimate_price":
        content = estimate_price.invoke(tool_args)
    elif tool_name == "web_search":
        content = web_search.invoke(tool_args)
    elif tool_name == "compare_prices":
        content = compare_prices.invoke(tool_args)
    else:
        content = f"Unknown tool: {tool_name}"

    return ToolMessage(content=str(content), tool_call_id=tool_call_id)


def convert_history_to_messages(history: List[dict]) -> List[BaseMessage]:
    """Convert Gradio chat history into LangChain message objects."""
    messages: List[BaseMessage] = []
    for item in history:
        role = item.get("role", "")
        content = str(item.get("content", ""))
        if role == "user":
            messages.append(HumanMessage(content=content))
        elif role == "assistant":
            messages.append(AIMessage(content=content))
    return messages


def run_agent(user_text: str, history: Optional[List[dict]] = None) -> str:
    """Run the tool-calling agent loop and return the final assistant response."""
    if history is None:
        history = []

    messages: List[BaseMessage] = [SystemMessage(content=SYSTEM_PROMPT.strip())]
    messages.extend(convert_history_to_messages(history))
    messages.append(HumanMessage(content=user_text))

    for _ in range(6):
        ai_message = agent_llm.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            return (
                ai_message.content
                if isinstance(ai_message.content, str)
                else str(ai_message.content)
            )

        for tool_call in ai_message.tool_calls:
            messages.append(handle_tool_call(tool_call))

    return "I could not complete the request after several steps. Please try rephrasing."

In [5]:
def chat_handler(user_text: str, history: List[dict]):
    """Handle a chat turn: run the agent and update history."""
    if history is None:
        history = []

    user_text = (user_text or "").strip()
    if not user_text:
        return history, ""

    response = run_agent(user_text, history)
    updated_history = history + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": response},
    ]
    return updated_history, ""


def build_gradio_app() -> gr.Blocks:
    """Build the Gradio UI for the Smart Shopping Assistant."""
    with gr.Blocks(title="Smart Shopping Assistant") as demo:
        gr.Markdown("# Smart Shopping Assistant")
        gr.Markdown(
            "Describe a product you're considering buying. "
            "The assistant will estimate its fair market value, search for real listings, "
            "and tell you if it's a good deal."
        )
        gr.Markdown(
            "**Example:** *Is a Sony WH-1000XM5 for $280 a good deal?* "
            "or *What's a fair price for an Apple Watch Series 9?*"
        )

        chatbot = gr.Chatbot(label="Shopping Assistant", height=450)
        user_input = gr.Textbox(
            label="Your question",
            placeholder="Describe a product or ask about a price...",
        )
        send_btn = gr.Button("Send", variant="primary")

        send_btn.click(
            fn=chat_handler,
            inputs=[user_input, chatbot],
            outputs=[chatbot, user_input],
        )
        user_input.submit(
            fn=chat_handler,
            inputs=[user_input, chatbot],
            outputs=[chatbot, user_input],
        )

    return demo


demo_app = build_gradio_app()

In [ ]:
demo_app.launch(share=True)